In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import pandas as pd
import pickle
from tqdm.notebook import tqdm
import polars as pl
import xgboost as xgb
print("xgboost version:", xgb.__version__)

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location

from src.plot_helpers import make_histogram_plot

from src.ntuple_variables.variables import combined_training_vars

from src.df_helpers import lazy_height


# File Loading

In [ ]:
print("loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"num events in all_df: {lazy_height(all_df)}")


In [ ]:
full_pred_data = all_df.filter(
    ~pl.col("filetype").is_in(["isotropic_one_gamma_overlay", "delete_one_gamma_overlay"])
)

presel_merged_df_allvars = full_pred_data.filter(pl.col("wc_kine_reco_Enu") > 0)

presel_merged_df_allvars = presel_merged_df_allvars.filter(pl.col("normalizing_run_period") != "4a")


In [ ]:
presel_merged_df_allvars = presel_merged_df_allvars.with_columns([
    (
        (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
        (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
        (pl.col("wc_single_photon_numu_score") > 0.4) &
        (pl.col("wc_single_photon_other_score") > 0.2) &
        (pl.col("wc_single_photon_ncpi0_score") > -0.05) &
        (pl.col("wc_single_photon_nue_score") > -1.0) &
        (pl.col("wc_shw_sp_n_20br1_showers") == 1)
    ).cast(pl.Int32).alias("erin_inclusive_1g_sel"),
    (
        (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
        (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
        (pl.col("wc_single_photon_numu_score") > 0.4) &
        (pl.col("wc_single_photon_other_score") > 0.2) &
        (pl.col("wc_single_photon_ncpi0_score") < -0.05)
    ).cast(pl.Int32).alias("erin_nc_pi0_sideband_flag"),
])


# Preselection Histogram

In [ ]:
# load columns from presel_merged_df
load_vars = list(presel_merged_df_allvars.collect_schema().names())

# remove columns combined_training_vars variables, tons of variables that aren't needed
load_vars = [col for col in load_vars if not (col in combined_training_vars)]

load_vars += [col for col in load_vars if ("blip" in col) and (col not in load_vars)]

# TEMPORARY, since we didn't exclude all the pandora postprocessing variables
#load_vars = [col for col in load_vars if not (["pandora_max" in col])]

extra_vars = [
    "wc_kine_reco_Enu",
    "wc_reco_num_protons_35_MeV",
    "wc_reco_backwards_projected_dist",
    "wc_reco_distance_to_boundary",
    "wc_reco_shower_theta",
    "wc_reco_shower_phi",
    "wc_kine_pio_mass",
    "lantern_diphoton_mass",
]

# add back in the current variable
for var in extra_vars:
    if var not in load_vars:
        load_vars.append(var)


In [ ]:
#load_vars

In [ ]:
presel_merged_df = presel_merged_df_allvars.select(load_vars).collect()

presel_merged_df

In [ ]:
make_histogram_plot(pred_and_data_sel_df=presel_merged_df, bins=np.linspace(0, 2000, 21), 
            var="wc_kine_reco_Enu", display_var=r"WC Reconstructed $E_\nu$ (MeV)", title="Preselection",
            selname="generic_presel",
            data_type="4b open data"
            )


# Erin Inclusive 1g Selection Histograms

## Normal

In [ ]:
erin_1g_sel_df = presel_merged_df.filter(pl.col("erin_inclusive_1g_sel") == 1)


In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV", display_var=r"WC Reconstructed num protons (35 MeV threshold)", title="Erin Inclusive 1g Selection",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            )


## With CRT cut

In [ ]:
erin_1g_sel_crt_df = erin_1g_sel_df.filter(pl.col("pandora_crtveto") == 0)


In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV", display_var=r"WC Reconstructed num protons (35 MeV threshold)", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            )


## With Enhanced 0p/Np split

In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_backtrack_cones_n", display_var=r"blip_backtrack_cones_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_no_shower_cone_no_backtrack_cones_n", display_var=r"blip_no_shower_cone_no_backtrack_cones_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_sphere_n", display_var=r"blip_sphere_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_no_shower_cone_n", display_var=r"blip_no_shower_cone_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
            

In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 4, 5), include_overflow=True,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            )


## With Nn/0n split

In [ ]:
erin_1g_sel_crt_0n_df = erin_1g_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2)
                                 & (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11))
erin_1g_sel_crt_Nn_df = erin_1g_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2)
                                 | (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11))



In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_Nn_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin Inclusive 1g Selection with CRT veto and Nn blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            )

make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_0n_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin Inclusive 1g Selection with CRT veto and 0n blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            )

In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_Nn_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin Inclusive 1g Selection with CRT veto and Nn blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0pNn0n_categories",
            legend_fontsize=10, legend_ncol=1,
            )

make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_0n_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin Inclusive 1g Selection with CRT veto and 0n blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0pNn0n_categories",
            legend_fontsize=10, legend_ncol=1,
            )

# Erin Sideband Histograms

In [ ]:
erin_ncpi0_sel_df = presel_merged_df.filter(pl.col("erin_nc_pi0_sideband_flag") == 1)
erin_ncpi0_sel_crt_df = erin_ncpi0_sel_df.filter(pl.col("pandora_crtveto") == 0)

erin_ncpi0_sel_crt_0n_df = erin_ncpi0_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2)
                                 & (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11))
erin_ncpi0_sel_crt_Nn_df = erin_ncpi0_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2)
                                 | (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11))


In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_Nn_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin NC Pi0 Sideband with CRT veto and Nn blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 240],
            )

make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_0n_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin NC Pi0 Sideband with CRT veto and 0n blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 240],
            )

In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_Nn_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin NC Pi0 Sideband with CRT veto and Nn blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0pNn0n_pi0_categories",
            legend_fontsize=10, legend_ncol=1,
            )

make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_0n_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="wc_reco_num_protons_35_MeV_plus_backtrack_blips", display_var=r"WC Reconstructed num protons (35 MeV threshold) + backtrack blips", title="Erin NC Pi0 Sideband with CRT veto and 0n blip cut",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0pNn0n_pi0_categories",
            legend_fontsize=10, legend_ncol=1,
            )